# 03 · sklearn pipeline — quantile gradient boosting

Features as specified: day of week, hour of day on a circle (`sin`/`cos`, three
harmonics so the sharp 08:00 edge is representable), and recent activity at nine lags
— 5, 15, 30, 60 min, 2, 4, 8, 24, 48 h — for burn, for human presence, and for
whether the machine was even awake.

`loss="quantile"` predicts a quantile directly, which under this cost model is
**exactly the right loss**: minimising `C_stolen × shortfall + C_wasted × excess` *is*
quantile regression at `q = 0.909`. So `quantile` is tuned as a hyper-parameter and we
can check whether the empirical optimum agrees with the theory.

`HistGradientBoostingRegressor`, not `GradientBoostingRegressor`: same loss, ~50×
faster, which is what makes the sweep affordable.

**Deployment constraint, kept in view:** a tree ensemble cannot be exported to a
handful of coefficients, so shipping this means evaluating trees in pure Python inside
a 5-minute tick, or `release/` stops being standard-library only.

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, str(pathlib.Path.cwd() / "lab"))
import numpy as np, pandas as pd
import dataset as D

frame = D.build("claude")
train, validation, test = D.split(frame)
COLS = D.features(frame)
print("\ncost model:  stolen $%.2f/pct   wasted $%.2f/pct   ratio %.0f:1"
      % (D.COST_STOLEN, D.COST_WASTED, D.COST_STOLEN / D.COST_WASTED))
print("the cost-optimal quantile is therefore %.4f" % D.optimal_quantile())
from sklearn.ensemble import HistGradientBoostingRegressor

12075 anchors over 41.9 days, embargo 300 min at each seam
  train        7185 rows  08-24 17:55 -> 09-18 16:35
  validation   2355 rows  09-18 21:40 -> 09-27 01:50
  test         2415 rows  09-27 06:55 -> 10-05 16:05
  dropped to the embargo: 120

cost model:  stolen $3.20/pct   wasted $0.32/pct   ratio 10:1
the cost-optimal quantile is therefore 0.9091


## The objective

Not accuracy — **money**. Two unit costs per percent of a 5-hour meter:

| | $/pct | Meaning |
|---|---|---|
| `COST_WASTED` | **0.32** | quota the bot was refused although the human never came — exactly 1% of a $32 unit |
| `COST_STOLEN` | **3.20** | quota the human wanted and the bot took — 10×, because the user has first claim |

Hyper-parameters below are chosen on **validation**, by cost. The test set is read
only in `05_comparison.ipynb`.

In [2]:
def fit_gbr(fit_frame, N, q, **kw):
    m = ~np.isnan(fit_frame["y%d" % N].values)
    g = HistGradientBoostingRegressor(loss="quantile", quantile=q, random_state=0, **kw)
    g.fit(fit_frame.loc[m, COLS], fit_frame.loc[m, "y%d" % N])
    return g

CAPACITY = {"small  d3  200": dict(max_depth=3, max_iter=200, learning_rate=0.05),
            "medium d6  400": dict(max_depth=6, max_iter=400, learning_rate=0.05),
            "large  d10 400": dict(max_depth=10, max_iter=400, learning_rate=0.05)}
QS = (0.75, 0.85, 0.909, 0.95)
cache = {}
candidates = {}
for cl, kw in CAPACITY.items():
    for q in QS:
        candidates["%s q=%.3f" % (cl, q)] = (cl, q)
        for N in D.HORIZONS:
            cache[(cl, q, N)] = fit_gbr(train, N, q, **kw)
print("fitted %d models" % len(cache))

fitted 60 models


In [3]:
chosen, table = D.tune("sklearn quantile", candidates, validation,
    lambda c, N: cache[(c[0], c[1], N)].predict(validation[COLS]), show=4)

N = 30   best: small  d3  200 q=0.909       $0.4470   (worst $1.9965)
                config  cost_usd  stolen_usd  wasted_usd
small  d3  200 q=0.909    0.4470      0.3650      0.0820
medium d6  400 q=0.909    0.4476      0.3801      0.0674
large  d10 400 q=0.909    0.4577      0.4066      0.0512
medium d6  400 q=0.850    0.4606      0.4291      0.0315

N = 60   best: small  d3  200 q=0.850       $1.5481   (worst $8.0704)
                config  cost_usd  stolen_usd  wasted_usd
small  d3  200 q=0.850    1.5481      1.5157      0.0324
large  d10 400 q=0.750    1.5882      1.5674      0.0209
large  d10 400 q=0.850    1.6237      1.6027      0.0210
medium d6  400 q=0.750    1.6471      1.5747      0.0724

N = 120  best: large  d10 400 q=0.950       $3.1065   (worst $8.3517)
                config  cost_usd  stolen_usd  wasted_usd
large  d10 400 q=0.950    3.1065      1.7439      1.3626
small  d3  200 q=0.850    3.6160      1.1413      2.4747
large  d10 400 q=0.909    4.0675      2.1510   

In [4]:
t = table.copy()
t["q"] = [float(s.split("q=")[1]) for s in t["config"]]
print("cheapest cost per quantile, per horizon  (theory: q = %.3f)" % D.optimal_quantile())
print(t.groupby(["N", "q"])["cost_usd"].min().unstack().round(3).to_string())

cheapest cost per quantile, per horizon  (theory: q = 0.909)
q    0.750  0.850  0.909  0.950
N                              
30   0.519  0.461  0.447  0.924
60   1.588  1.548  2.155  4.785
120  4.087  3.616  4.068  3.106
180  7.099  5.161  5.355  6.057
300  9.196  7.503  7.675  9.101


In [5]:
val = [D.evaluate("sklearn quantile", validation["y%d" % N].values,
                  cache[(chosen[N][1][0], chosen[N][1][1], N)].predict(validation[COLS]))
       | {"N": N} for N in D.HORIZONS]
print(pd.DataFrame(val).set_index("N")[["cost_usd", "stolen", "wasted", "coverage"]].round(3))
print()
cl3, q3 = chosen[300][1]
print(D.by_cell(validation, validation["y300"].values,
                {"sklearn": cache[(cl3, q3, 300)].predict(validation[COLS])}
                ).to_string(index=False))
D.save("03_sklearn_quantile",
       {"kind": "hist_quantile",
        "per_horizon": {str(N): {"q": chosen[N][1][1], **CAPACITY[chosen[N][1][0]]}
                        for N in D.HORIZONS}}, val)

     cost_usd  stolen  wasted  coverage
N                                      
30      0.447   0.114   0.256    91.635
60      1.548   0.474   0.101    87.219
120     3.106   0.545   4.258    94.650
180     5.161   1.010   6.025    91.677
300     7.503   1.371   9.734    87.389

 state      band   n  ACTUAL p95  sklearn
  idle     night 759         5.0      7.2
  idle   morning 421        38.0     29.8
  idle afternoon 253        40.0     28.0
  idle   evening 479         0.0      8.4
  warm     night  20         3.0     16.3
  warm   morning  28        83.0     43.1
  warm afternoon  62        84.9     47.0
  warm   evening  92         0.0     20.5
active     night  12         3.0     14.0
active   morning  31        79.0     58.5
active afternoon 165        84.0     48.9
active   evening  33         5.4     28.2
froze 03_sklearn_quantile -> design/02_prediction/lab/results/03_sklearn_quantile.json


PosixPath('/Users/jeanlescut/dev/agent-quota-maximizer/design/02_prediction/lab/results/03_sklearn_quantile.json')